In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
if os.environ['api_key']:
    print("API key is set.")

API key is set.


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA


llm = ChatNVIDIA(
  model="openai/gpt-oss-20b",
  api_key="xxx", 
  temperature=0,
  top_p=1,
  max_completion_tokens=4096,
)


In [4]:
response = llm.invoke("What is ai tell me in one line")
print(response.content)

Artificial intelligence (AI) is the field of computer science that creates systems capable of performing tasks that normally require human intelligence, such as learning, reasoning, and problem‑solving.


## **Rag implemnattaion with own text data **





### step 1: prepare document for the text


In [5]:
from langchain_core.documents import Document

text= """ High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]

Artificial intelligence was founded as an academic discipline in 1956.[4] The field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Funding and interest increased substantially after 2012, when graphics processing units (GPUs) started being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[9] This growth accelerated further after 2017 with the transformer architecture.[10] The 2020s AI boom was spurred on largely by advances in generative AI, which can create and modify media. In addition to AI safety and unintended consequences and harms from the use of AI, ethical concerns, AI's long-term effects, environmental effects,[11] and potential existential risks have prompted discussions of AI regulation. The symbolic approach has difficulty with several problems: the breadth of commonsense knowledge (the set of atomic facts the average person knows is enormous),[30] the sub-symbolic form of most commonsense knowledge (much of what people know is not represented as "facts" or "statements" they can express verbally),[15] and knowledge acquisition (the problem of obtaining knowledge for AI applications).[e]

Large language models (and some other AI programs developed since 2012) do not require explicit, symbolic knowledge. They acquire knowledge by being trained on the combined text of millions of books and billions of websites. Modern AI can also learn about a domain by running experiments (as when AlphaZero learns game strategy by playing against itself). Machine learning solves the problems of general knowledge, commonsense knowledge and knowledge acquisition, however, this approach has struggled with accurate recall and valid reasoning"""

In [6]:
docs = [Document(page_content=text, metadata={"source": "Wikipedia", "DocumentID": "Doc1"})]

# ** chunking the document into smaller pieces **

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter( chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'Wikipedia', 'DocumentID': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).'),
 Document(metadata={'source': 'Wikipedia', 'DocumentID': 'Doc1'}, page_content='The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2]'),
 Document(metadata={'source': 'Wikipedia', 'DocumentID': 'Doc1'}, page_content='philosophy, 

## Embeddings creation fro chunks

In [ ]:
import os
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

api_key = os.environ.get("api_key", "xxx")

embedding_model = NVIDIAEmbeddings(
    model="nvidia/nemotron-3.5-embed-1b",
    nvidia_api_key=api_key
)


d:\rag_lectu\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:269: UserWarning: Found nvidia/nemotron-3.5-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


## vector store and vectore creation

In [9]:
from langchain_community.vectorstores import Chroma

vector = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model)



C:\Users\sathwik\AppData\Local\Temp\ipykernel_14248\2651872665.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


#### ** DO the search to read the data**

In [10]:
context = vector.similarity_search("What is AI", k=3)

### read data to the context through llm

In [11]:
response = llm.invoke(f"what is AI ? you can answer using the context:{context}")
print(response.content)

Artificial Intelligence (AI) is a branch of computer science that seeks to build systems capable of performing tasks that normally require human intelligence.  According to the provided sources, its traditional goals include:

| Goal | What it means |
|------|---------------|
| **Learning** | Acquiring knowledge or skills from data or experience |
| **Reasoning** | Drawing logical conclusions or making decisions |
| **Knowledge representation** | Storing and organizing information in a machine‑readable form |
| **Planning** | Sequencing actions to achieve a goal |
| **Natural language processing** | Understanding and generating human language |
| **Perception** | Interpreting sensory data (vision, sound, etc.) |
| **Robotics support** | Enabling autonomous physical agents |

To reach these goals, AI researchers employ a mix of techniques such as:

* **State‑space search** and **mathematical optimisation**  
* **Formal logic** and symbolic reasoning  
* **Artificial neural networks** (d

## ** Reading through PDF data on fly**

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(
  model="openai/gpt-oss-20b",
    api_key="xxx",
    temperature=0,
     timeout=360
)


In [73]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('D:\\rag_lectu\\.venv\\Document\\Dataengineer.pdf')
pdf = loader.load()

pdf


[Document(metadata={'producer': 'calibre 6.28.1', 'creator': 'calibre 6.28.1', 'creationdate': '2024-03-16T13:58:44+00:00', 'author': 'Verma, Sumit', 'moddate': '2024-03-16T13:58:44+00:00', 'title': 'Data Engineering with Databricks', 'source': 'D:\\rag_lectu\\.venv\\Document\\Dataengineer.pdf', 'total_pages': 219, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'calibre 6.28.1', 'creator': 'calibre 6.28.1', 'creationdate': '2024-03-16T13:58:44+00:00', 'author': 'Verma, Sumit', 'moddate': '2024-03-16T13:58:44+00:00', 'title': 'Data Engineering with Databricks', 'source': 'D:\\rag_lectu\\.venv\\Document\\Dataengineer.pdf', 'total_pages': 219, 'page': 1, 'page_label': '2'}, page_content='Contents\nWhat is Databricks\nUse cases for Azure Databricks\nBuild an enterprise data Lakehouse\nETL and Data Engineering\nMachine learning, AI, and Data science\nData warehousing, Analytics, and BI\nLarge language models and generative AI\nData governance\nDevOps, CI/CD

## changing metadat custom

In [74]:
for i in pdf:
    i.metadata={"source": "Dataengineer.pdf", "author": "Verma, Sumit"}
    

## splitting docuemts to chunks

In [75]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter( chunk_size=1000, chunk_overlap=50)

chunks=splitter.split_documents(pdf)
chunks

[Document(metadata={'source': 'Dataengineer.pdf', 'author': 'Verma, Sumit'}, page_content='Contents\nWhat is Databricks\nUse cases for Azure Databricks\nBuild an enterprise data Lakehouse\nETL and Data Engineering\nMachine learning, AI, and Data science\nData warehousing, Analytics, and BI\nLarge language models and generative AI\nData governance\nDevOps, CI/CD, and task orchestration\nReal-time and streaming analytics\nDatabricks Lakehouse\nMedallion Lakehouse Architecture\nBronze layer\nSilver Layer\nGold Layer\nData objects in the Databricks Lakehouse\nDatabricks architecture\nLoad data\nAuto Loader\nExternal Data\nDelta Live Tables\nDelta Live Tables datasets\nStreaming table\nMaterialized view\nViews\nCreate Delta live pipeline\nPipeline update\nSchedule a pipeline'),
 Document(metadata={'source': 'Dataengineer.pdf', 'author': 'Verma, Sumit'}, page_content='Data quality\nMaintenance tasks\nDevelopment and production modes\nPublish data to Hive metastore\nPublish data to Unity Cata

## embeedings

In [ ]:
import os
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings


api_key = os.environ.get("api_key", "xxx")

embedding_model = NVIDIAEmbeddings(
    model="nvidia/nemotron-3.5-embed-1b",
    nvidia_api_key=api_key
)


d:\rag_lectu\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:269: UserWarning: Found nvidia/nemotron-3.5-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


##VECTOR STORE

In [81]:
from langchain_community.vectorstores.chroma import Chroma

vector= Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model)

In [82]:
vector.similarity_search(("display information about the data lakehouse ?"),k=3)

[Document(metadata={'source': 'D:\\rag_lectu\\.venv\\Document\\Dataengineer.pdf', 'title': 'Data Engineering with Databricks', 'page': 14, 'total_pages': 219, 'moddate': '2024-03-16T13:58:44+00:00', 'producer': 'calibre 6.28.1', 'author': 'Verma, Sumit', 'creator': 'calibre 6.28.1', 'page_label': '15', 'creationdate': '2024-03-16T13:58:44+00:00'}, page_content='Databricks Lakehouse\nA data Lakehouse is a new, open data management\narchitecture that combines the ﬂexibility, cost-eﬃciency,\nand scale of\xa0 Data Lake and ACID transactions of data\nwarehouses. It enables business intelligence (BI) and\nmachine learning (ML) on all data.\n\xa0\xa0\nData Lakehouses are enabled by\xa0combining data structures\nand data management features of Data warehouse and low-\ncost storage used for Data lakes. Merging data warehouse &\nData lakes together into a single system helps move data\nteam faster. Data Lakehouse also ensures that teams have\nthe most complete and up-to-date data available for d

## read through the LLM

In [79]:
context = llm.invoke(f"what is data lakehouse expalin with context : {context}")
print(context.content)